In [1]:
import pandas as pd
import numpy as np

## Historical Analysis Calculation

In [ ]:
# ── CLEANING FUNCTION ───────────────────────────────────────────────────────
# Investing.com exports have formatting issues we need to fix:
# - Dates come as strings e.g. "02/26/2015"
# - Prices have commas e.g. "4,686.19"
# - Volume has suffixes e.g. "62.82M" or "1.2B"
# - Change % has a percent sign e.g. "1.19%"
# This function standardises all of that into clean numeric columns

def clean_investing_csv(df, keep_ohlc=False, keep_price=False):
    df = df.copy()
    
    # Parse date and sort oldest to newest
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Clean and convert Price column
    df['Price'] = df['Price'].astype(str).str.replace(',', '').astype(float)
    
    # Optionally keep OHLC columns
    if keep_ohlc:
        for col in ['Open', 'High', 'Low']:
            df[col] = df[col].astype(str).str.replace(',', '').astype(float)
    
    # Convert volume suffixes
    def parse_volume(v):
        v = str(v).strip().replace(',', '')
        if v.endswith('B'):
            return float(v[:-1]) * 1_000_000_000
        elif v.endswith('M'):
            return float(v[:-1]) * 1_000_000
        elif v.endswith('K'):
            return float(v[:-1]) * 1_000
        else:
            try:
                return float(v)
            except:
                return np.nan
    
    df['Vol.'] = df['Vol.'].apply(parse_volume)
    
    # Calculate simple daily return from Price
    df['return'] = df['Price'].pct_change()
    
    # Build final column list based on flags
    cols = ['Date', 'return', 'Vol.']
    if keep_price:
        cols.append('Price')
    if keep_ohlc:
        cols += ['Open', 'High', 'Low']
    
    return df[cols].copy()

In [ ]:
# ── LOAD AND CLEAN BOTH UAE EXCHANGES ──────────────────────────────────────
# Both need keep_price=True so we can compute the weighted combination
# return is already calculated inside clean_investing_csv so no need to repeat it

uae_adx = clean_investing_csv(
    pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\FTSE ADX General Historical Data.csv"),
    keep_price=True
)
uae_dfm = clean_investing_csv(
    pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\DFM General Historical Data.csv"),
    keep_price=True
)

print(uae_adx.columns.tolist())
print(uae_dfm.columns.tolist())

# ── MERGE ADX AND DFM ON DATE ──────────────────────────────────────────────
uae = pd.merge(
    uae_adx[['Date', 'Price', 'return', 'Vol.']],
    uae_dfm[['Date', 'Price', 'return', 'Vol.']],
    on='Date',
    suffixes=('_adx', '_dfm')
)

# ── BUILD COMBINED UAE INDEX ────────────────────────────────────────────────
ADX_WEIGHT = 0.70
DFM_WEIGHT = 0.30

uae['return']    = (ADX_WEIGHT * uae['return_adx']) + (DFM_WEIGHT * uae['return_dfm'])
uae['Vol.']      = uae['Vol._adx'] + uae['Vol._dfm']
uae['Price']     = 1000 * np.exp(uae['return'].cumsum())

# ── FINAL CLEAN UAE DATAFRAME ──────────────────────────────────────────────
uae_combined = uae[['Date', 'return', 'Vol.', 'Price']].copy()
uae_combined = uae_combined.dropna().reset_index(drop=True)

print(f"UAE combined: {len(uae_combined)} trading days")
print(f"Date range: {uae_combined['Date'].min().date()} to {uae_combined['Date'].max().date()}")
print(uae_combined.head())

['Date', 'return', 'Vol.', 'Price']
['Date', 'return', 'Vol.', 'Price']
UAE combined: 748 trading days
Date range: 2012-03-04 to 2015-02-26
        Date    return         Vol.        Price
0 2012-03-04  0.013846  946180000.0  1013.942763
1 2012-03-05 -0.004194  944040000.0  1009.699265
2 2012-03-06 -0.015226  709770000.0   994.441564
3 2012-03-07 -0.029676  588610000.0   965.364324
4 2012-03-08  0.002814  475060000.0   968.084721


In [7]:
# ── CLEAN QATAR, SAUDI AND MSCI EM ────────────────────────────────────────────
# These are single-index countries so no combining needed
# Default keep_price=False — Price column dropped after return is calculated
# msci_em Vol. column not used for analysis but kept for structural consistency

qatar   = clean_investing_csv(pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\QE All Shares Historical Data.csv"))
saudi   = clean_investing_csv(pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\Tadawul All Share Historical Data.csv"))
msci_em = clean_investing_csv(pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\MSCI Emerging Markets Historical Data.csv"))

print(f"Qatar:   {len(qatar)} days | {qatar['Date'].min().date()} to {qatar['Date'].max().date()}")
print(f"Saudi:   {len(saudi)} days | {saudi['Date'].min().date()} to {saudi['Date'].max().date()}")
print(f"MSCI EM: {len(msci_em)} days | {msci_em['Date'].min().date()} to {msci_em['Date'].max().date()}")

Qatar:   744 days | 2012-03-01 to 2015-02-26
Saudi:   684 days | 2017-09-06 to 2020-05-31
MSCI EM: 2329 days | 2012-01-30 to 2020-12-31


In [8]:
print(uae_combined.columns.tolist())
print(qatar.columns.tolist())
print(saudi.columns.tolist())
print(msci_em.columns.tolist())

['Date', 'return', 'Vol.', 'Price']
['Date', 'return', 'Vol.']
['Date', 'return', 'Vol.']
['Date', 'return', 'Vol.']


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
# WINDOW = 126 trading days = approximately 6 calendar months
# All four windows use the same length for comparability

WINDOW = 126
OUTPUT_PATH = r"C:\Users\USER\Desktop\Arqaam_Intern_Case\\"

events = {
    "UAE": {
        "df": uae_combined,
        "announcement": pd.Timestamp("2013-06-11"),
        "inclusion":    pd.Timestamp("2014-05-31"),
    },
    "Qatar": {
        "df": qatar,
        "announcement": pd.Timestamp("2013-06-11"),
        "inclusion":    pd.Timestamp("2014-05-31"),
    },
    "Saudi": {
        "df": saudi,
        "announcement": pd.Timestamp("2018-06-20"),
        "inclusion":    pd.Timestamp("2019-05-28"),
    },
}


# ══════════════════════════════════════════════════════════════════════════════
# WINDOW SLICER
# ══════════════════════════════════════════════════════════════════════════════
# Locates announcement and inclusion dates in the dataframe by index position
# then slices four equal 126-day windows around each event date

def get_windows(df, ann, inc, window):
    ann_idx = df[df['Date'] >= ann].index[0]
    inc_idx = df[df['Date'] >= inc].index[0]
    
    pre         = df.iloc[ann_idx - window : ann_idx].copy()
    early_trans = df.iloc[ann_idx           : ann_idx + window].copy()
    late_trans  = df.iloc[inc_idx - window  : inc_idx].copy()
    post        = df.iloc[inc_idx           : inc_idx + window].copy()
    
    return {
        "Pre-Announcement": pre,
        "Early Transition": early_trans,
        "Late Transition":  late_trans,
        "Post-Inclusion":   post
    }


# ══════════════════════════════════════════════════════════════════════════════
# PRECOMPUTE ROLLING CORRELATION ON FULL DATASET
# ══════════════════════════════════════════════════════════════════════════════
# Rolling correlation must be calculated on the full dataset BEFORE slicing
# into windows. If calculated per window, the first 60 rows of each window
# produce NaN because the rolling window has no prior history to work with.
# Calculating on the full dataset first ensures every date has a valid value
# as long as it is at least 60 trading days from the start of the full series.

def precompute_rolling_corr(df, msci_em, window=60):
    # Merge country returns with MSCI EM returns on matching dates
    merged = pd.merge(
        df[['Date', 'return']],
        msci_em[['Date', 'return']].rename(columns={'return': 'msci_return'}),
        on='Date', how='inner'
    )
    # 60-day rolling pearson correlation
    merged['Rolling_Corr_60d'] = merged['return'].rolling(window).corr(merged['msci_return'])
    
    return merged[['Date', 'Rolling_Corr_60d']]


# ══════════════════════════════════════════════════════════════════════════════
# METRIC CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════════
# Takes the four sliced windows and precomputed correlation series
# Returns one summary row per window with all four metrics
#
# Metric 1 — Cumulative Return (%)
#   Total simple return over the window if held from start to end
#
# Metric 2 — Avg Daily Volume and Volume Uplift vs Pre-Announcement (%)
#   Average daily traded value per window
#   Uplift = % change vs pre-announcement baseline
#
# Metric 3 — Annualised Volatility (%)
#   Std dev of daily returns scaled to annual by multiplying by sqrt(252)
#
# Metric 4 — Avg Rolling Correlation with MSCI EM
#   Average of the precomputed 60-day rolling correlation within each window
#   NaNs dropped so incomplete rolling windows don't distort the average

def calc_metrics(windows, corr_series):
    results = {}
    pre_avg_vol = windows["Pre-Announcement"]['Vol.'].mean()
    
    for window_name, df in windows.items():
        
        # ── Metric 1: Cumulative Return ────────────────────────────────────────
        cum_return_pct = round(((1 + df['return']).prod() - 1) * 100, 2)
        
        # ── Metric 2: Volume ───────────────────────────────────────────────────
        avg_daily_vol  = df['Vol.'].mean()
        vol_uplift_pct = round(((avg_daily_vol / pre_avg_vol) - 1) * 100, 2)
        
        # ── Metric 3: Volatility ───────────────────────────────────────────────
        ann_vol_pct = round(df['return'].std() * np.sqrt(252) * 100, 2)
        
        # ── Metric 4: Correlation ──────────────────────────────────────────────
        # Slice the precomputed correlation series to this window's date range
        # dropna() removes any residual NaNs at the start of the full series
        window_corr = corr_series[
            (corr_series['Date'] >= df['Date'].min()) &
            (corr_series['Date'] <= df['Date'].max())
        ]['Rolling_Corr_60d'].dropna()
        
        avg_corr = round(window_corr.mean(), 3)
        
        results[window_name] = {
            "Cumulative Return (%)":        cum_return_pct,
            "Avg Daily Volume":             round(avg_daily_vol, 2),
            "Volume Uplift vs Pre-Ann (%)": vol_uplift_pct,
            "Annualised Volatility (%)":    ann_vol_pct,
            "Avg Rolling Correlation":      avg_corr,
        }
    
    return pd.DataFrame(results).T.reset_index().rename(columns={"index": "Window"})


# ══════════════════════════════════════════════════════════════════════════════
# ROLLING CORRELATION SERIES FOR EXCEL LINE CHART
# ══════════════════════════════════════════════════════════════════════════════
# Extracts the rolling correlation time series for the full study period
# Tags each row with its window label for reference in Excel
# Adds announcement and inclusion date columns for vertical line markers

def calc_rolling_corr_series(corr_series, df, ann, inc, window):
    ann_idx = df[df['Date'] >= ann].index[0]
    inc_idx = df[df['Date'] >= inc].index[0]
    
    # Define full study period: pre-announcement start to post-inclusion end
    start_date = df.iloc[ann_idx - window]['Date']
    end_date   = df.iloc[min(inc_idx + window, len(df) - 1)]['Date']
    
    # Slice the precomputed series to the study period
    result = corr_series[
        (corr_series['Date'] >= start_date) &
        (corr_series['Date'] <= end_date)
    ].copy()
    
    # Tag each row with its window label
    def tag_window(date):
        if date < ann:
            return "Pre-Announcement"
        elif date < ann + pd.DateOffset(months=6):
            return "Early Transition"
        elif date < inc:
            return "Late Transition"
        else:
            return "Post-Inclusion"
    
    result['Window']            = result['Date'].apply(tag_window)
    result['Announcement_Date'] = ann
    result['Inclusion_Date']    = inc
    
    return result[['Date', 'Rolling_Corr_60d', 'Window', 'Announcement_Date', 'Inclusion_Date']]


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP — RUN ALL COUNTRIES
# ══════════════════════════════════════════════════════════════════════════════
# For each country:
#   1. Precompute rolling correlation on full dataset
#   2. Slice four windows
#   3. Calculate all metrics and save summary
#   4. Save indexed price paths for line chart
#   5. Save volume data for bar charts
#   6. Save volatility summary for bar chart
#   7. Save rolling correlation series for line chart

for country, config in events.items():
    
    print(f"\nProcessing {country}...")
    
    df  = config["df"].copy().reset_index(drop=True)
    ann = config["announcement"]
    inc = config["inclusion"]
    
    # ── Step 1: Precompute rolling correlation on full dataset ─────────────────
    # Must happen before window slicing — see precompute_rolling_corr comments
    corr_series = precompute_rolling_corr(df, msci_em)
    
    # ── Step 2: Slice the four windows ────────────────────────────────────────
    windows = get_windows(df, ann, inc, WINDOW)
    
    # ── Step 3: Calculate all metrics and save summary ────────────────────────
    summary_df = calc_metrics(windows, corr_series)
    summary_df.to_excel(f"{OUTPUT_PATH}{country}_Summary.xlsx", index=False)
    print(f"  {country}_Summary.xlsx saved")
    
    # ── Step 4: Build indexed price paths ─────────────────────────────────────
    # Each window re-indexed to start at 100 so paths are directly comparable
    price_rows = []
    for window_name, wdf in windows.items():
        wdf = wdf.copy().reset_index(drop=True)
        wdf['Indexed_Price'] = 100 * (1 + wdf['return']).cumprod()
        wdf.loc[wdf.index[0], 'Indexed_Price'] = 100
        wdf['Window'] = window_name
        wdf['Day']    = range(1, len(wdf) + 1)
        price_rows.append(wdf[['Day', 'Window', 'Indexed_Price', 'Date']])
    
    price_df = pd.concat(price_rows, ignore_index=True)
    price_df.to_excel(f"{OUTPUT_PATH}{country}_Price.xlsx", index=False)
    print(f"  {country}_Price.xlsx saved")
    
    # ── Step 5: Save volume data ───────────────────────────────────────────────
    vol_rows = []
    pre_avg  = windows["Pre-Announcement"]['Vol.'].mean()
    
    for window_name, wdf in windows.items():
        wdf = wdf.copy().reset_index(drop=True)
        wdf['Window']            = window_name
        wdf['Avg_Daily_Volume']  = wdf['Vol.'].mean()
        wdf['Volume_Uplift_Pct'] = round(((wdf['Vol.'].mean() / pre_avg) - 1) * 100, 2)
        vol_rows.append(wdf[['Date', 'Window', 'Vol.', 'Avg_Daily_Volume', 'Volume_Uplift_Pct']])
    
    vol_df = pd.concat(vol_rows, ignore_index=True)
    vol_df.to_excel(f"{OUTPUT_PATH}{country}_Volume.xlsx", index=False)
    print(f"  {country}_Volume.xlsx saved")
    
    # ── Step 6: Save volatility summary ───────────────────────────────────────
    summary_df[['Window', 'Annualised Volatility (%)']].to_excel(
        f"{OUTPUT_PATH}{country}_Volatility.xlsx", index=False
    )
    print(f"  {country}_Volatility.xlsx saved")
    
    # ── Step 7: Save rolling correlation series ────────────────────────────────
    corr_df = calc_rolling_corr_series(corr_series, df, ann, inc, WINDOW)
    corr_df.to_excel(f"{OUTPUT_PATH}{country}_Correlation.xlsx", index=False)
    print(f"  {country}_Correlation.xlsx saved")

print("\nAll files saved successfully.")


Processing UAE...
  UAE_Summary.xlsx saved
  UAE_Price.xlsx saved
  UAE_Volume.xlsx saved
  UAE_Volatility.xlsx saved
  UAE_Correlation.xlsx saved

Processing Qatar...
  Qatar_Summary.xlsx saved
  Qatar_Price.xlsx saved
  Qatar_Volume.xlsx saved
  Qatar_Volatility.xlsx saved
  Qatar_Correlation.xlsx saved

Processing Saudi...
  Saudi_Summary.xlsx saved
  Saudi_Price.xlsx saved
  Saudi_Volume.xlsx saved
  Saudi_Volatility.xlsx saved
  Saudi_Correlation.xlsx saved

All files saved successfully.


## Calculating Equity Market Peformance Of Upgrade Candidates

In [ ]:
# ── LOAD FILES ──────────────────────────────────────────────────────────

msm   = pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\Thesis\MSCI Oman Historical Data.csv")
bse   = pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\Thesis\MSCI Bahrain Index.csv")
em    = pd.read_csv(r"C:\Users\USER\Desktop\Arqaam_Intern_Case\Thesis\MSCI Emerging Markets Historical Data (1).csv")

# ── PREVIEW — run this first to see your column names ──────────────────
print("=== MSCI Oman ===");   print(msm.head(3));   print(msm.columns.tolist())
print("=== MSCI Bahrain ===");  print(bse.head(3));   print(bse.columns.tolist())
print("=== MSCI Emerging Markets ===");  print(em.head(3));    print(em.columns.tolist())

=== MSCI Oman ===
         Date     Price      Open      High       Low  Vol. Change %
0  03/12/2026  1,213.09  1,203.67  1,222.05  1,200.17   NaN    0.74%
1  03/11/2026  1,204.21  1,212.98  1,218.82  1,200.37   NaN   -0.80%
2  03/10/2026  1,213.87  1,219.98  1,220.28  1,205.38   NaN   -0.68%
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
=== MSCI Bahrain ===
         Date   Close  % Change  % Change vs Average  Volume
0  12/29/2022  136.03      2.17                 2.14     NaN
1  12/30/2022  136.10      0.05                 0.02     NaN
2    1/1/2023  136.10      0.00                -0.03     NaN
['Date', 'Close', '% Change', '% Change vs Average', 'Volume']
=== MSCI Emerging Markets ===
         Date     Price      Open      High       Low  Vol. Change %
0  03/13/2026  1,469.47  1,484.88  1,489.90  1,465.48   NaN   -1.52%
1  03/12/2026  1,492.11  1,509.45  1,511.82  1,487.67   NaN   -1.61%
2  03/11/2026  1,516.47  1,504.44  1,532.95  1,504.44   NaN    0.81%
['Date', 'P

In [ ]:
# ── CLEAN EACH DATASET ──────────────────────────────────────────────────
def clean_price_df(df, date_col, price_col, market_name, dayfirst=False):
    out = df[[date_col, price_col]].copy()
    
    # clean date
    out[date_col] = pd.to_datetime(out[date_col], errors='coerce', dayfirst=dayfirst)
    
    # clean price
    out[price_col] = (
        out[price_col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    out[price_col] = pd.to_numeric(out[price_col], errors='coerce')
    
    # rename columns
    out.columns = ['Date', market_name]
    
    # sort oldest to newest
    out = out.dropna().sort_values('Date').reset_index(drop=True)
    
    return out

# Oman and MSCI EM use Price
msm_clean = clean_price_df(msm, 'Date', 'Price', 'Oman', dayfirst=False)
em_clean  = clean_price_df(em,  'Date', 'Price', 'MSCI_EM', dayfirst=False)

# Bahrain uses Close
bse_clean = clean_price_df(bse, 'Date', 'Close', 'Bahrain', dayfirst=False)

# ── 3. MERGE INTO ONE DATAFRAME ────────────────────────────────────────────
df = msm_clean.merge(bse_clean, on='Date', how='outer')
df = df.merge(em_clean, on='Date', how='outer')

# sort and set index
df = df.sort_values('Date').set_index('Date')

print(df.head())
print(df.tail())
print(df.info())

              Oman  Bahrain  MSCI_EM
Date                                
2021-12-01  613.86      NaN  1226.81
2021-12-02  610.10      NaN  1236.19
2021-12-03     NaN      NaN  1224.64
2021-12-06  613.08      NaN  1213.96
2021-12-07  618.73      NaN  1235.56
               Oman  Bahrain  MSCI_EM
Date                                 
2026-03-10  1213.87   186.91  1504.33
2026-03-11  1204.21   187.06  1516.47
2026-03-12  1213.09   186.91  1492.11
2026-03-13      NaN   186.85  1469.47
2026-03-15      NaN   184.63      NaN
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1286 entries, 2021-12-01 to 2026-03-15
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Oman     853 non-null    float64
 1   Bahrain  1005 non-null   float64
 2   MSCI_EM  1118 non-null   float64
dtypes: float64(3)
memory usage: 40.2 KB
None


In [44]:
# ── 4. RESAMPLE TO WEEKLY DATA ─────────────────────────────────────────────
# weekly is better for Bahrain/Oman because daily can be noisy and missing
weekly_prices = df.resample('W-FRI').last()

# forward-fill missing prices after resampling
weekly_prices = weekly_prices.ffill()

print(weekly_prices.head())
print(weekly_prices.tail())

              Oman  Bahrain  MSCI_EM
Date                                
2021-12-03  610.10      NaN  1224.64
2021-12-10  627.04      NaN  1238.54
2021-12-17  617.63      NaN  1216.30
2021-12-24  635.15      NaN  1220.55
2021-12-31  622.75      NaN  1232.01
               Oman  Bahrain  MSCI_EM
Date                                 
2026-02-20  1115.68   193.93  1567.23
2026-02-27  1139.73   193.56  1610.70
2026-03-06  1143.90   186.52  1499.72
2026-03-13  1213.09   186.85  1469.47
2026-03-20  1213.09   184.63  1469.47


In [51]:
# ── 5. WEEKLY RETURNS ──────────────────────────────────────────────────────
weekly_returns = weekly_prices.pct_change().dropna()

# risk-free rate assumption
rf = 0.037   # 3% month us tresury bill
weekly_rf = rf / 52

# helper functions
def trailing_return(price_series, years):
    end_date = price_series.dropna().index.max()
    start_date = end_date - pd.DateOffset(years=years)
    
    window = price_series.loc[price_series.index >= start_date].dropna()
    if len(window) < 2:
        return np.nan
    
    return (window.iloc[-1] / window.iloc[0]) - 1

def annualized_vol(return_series):
    return return_series.std() * np.sqrt(52)

def sharpe_ratio(return_series, annual_rf=0.05):
    ann_return = return_series.mean() * 52
    ann_vol = return_series.std() * np.sqrt(52)
    if ann_vol == 0:
        return np.nan
    return (ann_return - annual_rf) / ann_vol

# ── 6. BUILD SUMMARY TABLE ─────────────────────────────────────────────────
markets = ['Oman', 'Bahrain', 'MSCI_EM']
summary = []

for market in markets:
    price_series = weekly_prices[market].dropna()
    ret_series = weekly_returns[market].dropna()
    
    one_year = trailing_return(price_series, 1)
    three_year = trailing_return(price_series, 3)
    vol = annualized_vol(ret_series)
    sharpe = sharpe_ratio(ret_series, annual_rf=rf)
    
    if market != 'MSCI_EM':
        corr = ret_series.corr(weekly_returns['MSCI_EM'])
    else:
        corr = 1.0
    
    summary.append({
        'Market': market,
        '1Y Return': one_year,
        '3Y Return': three_year,
        'Volatility': vol,
        'Sharpe': sharpe,
        'Correlation vs MSCI EM': corr
    })

summary_df = pd.DataFrame(summary)

# format as percentages where relevant
summary_df['1Y Return'] = summary_df['1Y Return'] * 100
summary_df['3Y Return'] = summary_df['3Y Return'] * 100
summary_df['Volatility'] = summary_df['Volatility'] * 100

print(summary_df.round(2))

    Market  1Y Return  3Y Return  Volatility  Sharpe  Correlation vs MSCI EM
0     Oman      79.61      55.81       12.83    0.92                    0.13
1  Bahrain      17.76      35.90       11.34    0.56                    0.15
2  MSCI_EM      29.88      51.15       14.95    0.72                    1.00
